In [ ]:



import os
print(os.listdir('/content'))

['.config', 'rejected_rows.json', 'ecWvPsdV2FNmRZAa.csv', '.ipynb_checkpoints', 'clinic.csv', 'appoiments.csv', 'sample_data']


In [ ]:
import os
for file in os.listdir('/content'):
  if file.endswith('.csv'):
    print(file)

ecWvPsdV2FNmRZAa.csv
clinic.csv
appoiments.csv


In [ ]:
import pandas as pd
import os
for file in os.listdir('/content'):
  if file.endswith('.csv'):
    df=pd.read_csv('/content/'+file)
    print("Columns:",df.columns.tolist())
    print(df.head(3))

Columns: ['note_id', 'appointment_id', 'note_text']
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
  appointment_id patient_id clinic_id   booked_on   appt_date   age gen

In [ ]:
import os
import pandas as pd

for file in os.listdir('/content'):
  if file.endswith('.csv'):
    df=pd.read_csv('/content/'+file)
    print("\nFILE:",file)
    print("COLUMNS:",df.columns.tolist())


FILE: ecWvPsdV2FNmRZAa.csv
COLUMNS: ['note_id', 'appointment_id', 'note_text']

FILE: clinic.csv
COLUMNS: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

FILE: appoiments.csv
COLUMNS: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [ ]:
import csv
import json

def parse_row(row):
 row["age"]=int(row["age"])
 row["fee_inr"]=int(row["fee_inr"])
 row["prior_no_shows"]=int(row["prior_no_shows"])
 row["distance_km"]=float(row["distance_km"])
 return row

accepted, rejected=[],[]


with open("appointments.csv",newline="")as f:
    reader=csv.DictReader(f)
    print("Headers:",reader.fieldnames)

    for row in reader:
      try:
       accepted.append(parse_row(row))
      except ValueError:
       rejected.append(row)

    print("Total rows:",reader.line_num-1)

with open("rejected_rows.json","w") as f:
            json.dump(rejected,f,indent=2)

            print("Accepted:",len(accepted))
            print("Rejected:",len(rejected))

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted: 988
Rejected: 32


In [ ]:
import pandas as pd

#Load original data
df=pd.read_csv("appointments.csv")

#Initial inspection
print("initial shape:",df.shape)
df.info()
print("\nMissing values:\n",df.isnull().sum())
print("\nDuplicate rows:",df.duplicated().sum())

#(a) Remove exact duplicate rows
df=df.drop_duplicates()
print("\nAfter removing duplicates:",len(df))

#(b) Clean booking_channel
df["booking_channel"]=df["booking_channel"].str.strip().str.title()
print("After cleaning booking_channel:",len(df))

#(c)Convert date columns
df["appt_date"]=pd.to_datetime(df["appt_date"],
errors="coerce")
df["booked_on"]=pd.to_datetime(df["booked_on"])
print("After converting dates:",len(df))

#(d)Remove rows with missing appointmentsdate or status
df=df.dropna(subset=["appt_date","status"])
print("After dropping missing dates/status:",
      len(df))


initial shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
 appointment_id      0
patient_id          0
clin

In [ ]:
import numpy as np
import pandas as pd


df.loc[(df["age"]<1)|(df["age"]>100),"age"]=np.nan
df["age"]=df["age"].fillna(df["age"].median()).round().astype
(int)
df["distance_km"]=df["distance_km"].fillna(df["distance_km"].median())

for col in ["distance_km","fee_inr"]:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
IQR=Q3-Q1

lower=Q1-1.5*IQR
upper=Q3+1.5*IQR

capped=((df[col]<lower)|(df[col]>
                           upper)).sum()
df[col]=np.clip(df[col],lower,upper)

print(f"{col}:{capped}values capped")

df["distance_km"]=df["distance_km"].round(1)
df["fee_inr"]=df["fee_inr"].round().astype(int)

fee_inr:3values capped


In [ ]:
print(df[["age", "distance_km", "fee_inr", "lead_days", "no_show"]].dtypes)

age             object
distance_km    float64
fee_inr          int64
lead_days        int64
no_show          int64
dtype: object


In [ ]:
# Convert columns to numeric
for col in ["age", "distance_km", "fee_inr"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing values
df["age"] = df["age"].fillna(df["age"].median()).fillna(0).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median()).fillna(0)

df["fee_inr"] = df["fee_inr"].fillna(df["fee_inr"].median()).fillna(0)

In [ ]:

df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No_Show").astype(int)

X = df[["age", "distance_km", "fee_inr", "lead_days", "no_show"]].to_numpy(dtype=float)

print("Column means:", np.mean(X, axis=0))

count = (df["lead_days"] > 14).sum()
share = (df["lead_days"] > 14).mean()

print("Lead days >14 count:", count)
print("Lead days >14 share:", round(share, 3))

z = (df["distance_km"] - df["distance_km"].mean()) / df["distance_km"].std()

print("Distance z-score outliers:", (z.abs() > 2).sum())

Column means: [  0.          10.01707071 635.71414141  11.81919192   0.        ]
Lead days >14 count: 296
Lead days >14 share: 0.299
Distance z-score outliers: 8


In [ ]:

import pandas as pd
import numpy as np

# 1.4 Create Features

# Convert numeric columns
for col in ["age", "distance_km", "fee_inr"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Handle missing values
for col in ["age", "distance_km", "fee_inr"]:
    df[col] = df[col].fillna(df[col].median()).fillna(0)

df["age"] = df["age"].round().astype(int)

# Create new features
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

df["no_show"] = (
    df["status"] == "No_Show"
).astype(int)

# Create NumPy feature array
X = df[
    ["age", "distance_km", "fee_inr",
     "lead_days", "no_show"]
].to_numpy(dtype=float)

# Column means
print("Column means:", np.mean(X, axis=0))

# Lead days greater than 14
count = (df["lead_days"] > 14).sum()
share = (df["lead_days"] > 14).mean()

print("Lead days >14 count:", count)
print("Lead days >14 share:", round(share, 3))

# Distance Z-score outliers
z = (
    df["distance_km"] - df["distance_km"].mean()
) / df["distance_km"].std()

print("Distance z-score outliers:", (z.abs() > 2).sum())

Column means: [  0.          10.01707071 635.71414141  11.81919192   0.        ]
Lead days >14 count: 296
Lead days >14 share: 0.299
Distance z-score outliers: 8


In [ ]:
import pandas as pd
import requests

# Load clinic data
clinics = pd.read_csv("clinic.csv")

# Open-Meteo API
url = "https://api.open-meteo.com/v1/forecast"

temps = []

for _, clinic in clinics.iterrows():
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp = None
    reason = None

    # Try twice
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                reason = f"Status code {response.status_code}"
                continue

            data = response.json()

            if "hourly" not in data or not data["hourly"].get("temperature_2m"):
                reason = "Hourly temperature data missing"
                continue

            temp = data["hourly"]["temperature_2m"][0]
            reason = None
            break

        except requests.exceptions.RequestException as e:
            reason = str(e)

    if temp is None:
        print(clinic["clinic_name"], ":", reason)

    temps.append(temp)

# Add temperature column
clinics["temp_c"] = temps

# Save enriched data
clinics.to_csv("clinics_enriched.csv", index=False)

print("Enriched clinics:")
print(clinics)

print("Saved clinics_enriched.csv")

Enriched clinics:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    19.2  
1    19.5  
2    18.6  
3    14.4  
4    14.4  
5    27.9  
6    16.6  
7    14.0  
Saved clinics_enriched.csv


In [ ]:
import os

print("Current folder:", os.getcwd())
print("Available files:")
print(os.listdir("/content"))

Current folder: /content
Available files:
['.config', '.ipynb_checkpoints', 'appoiment.csv', 'clinic.csv', 'sample_data']


In [ ]:

import pandas as pd

df = pd.read_csv("appointments.csv")

print(df.shape)
print(df.head())

(1020, 14)
  appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1          A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2          A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3          A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
4          A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-in     Follow-up            No   
2          4.1      850         Walk-in  Consultation            No   
3          4.1      850         Walk-in  Consultation            No   
4          7.5      750             App  Consultation           Yes   

   prior_no_shows    status  
0               1   No-Show  
1               2  Attended  
2  

In [ ]:
print(df.columns.tolist())

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [ ]:
import pandas as pd
import numpy as np

# 1. Load appointment dataset
df = pd.read_csv("appointments.csv")

# 2. Convert date columns
df["booked_on"] = pd.to_datetime(df["booked_on"], errors="coerce")
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")

# 3. Create lead_days
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# 4. Create no_show target
df["no_show"] = (df["status"] == "No-Show").astype(int)

# 5. Select columns
columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "lead_days",
    "no_show"
]

# 6. Keep available columns
columns = [col for col in columns if col in df.columns]
clean = df[columns].copy()

# 7. Sort appointments
clean = clean.sort_values("appointment_id").reset_index(drop=True)

# 8. Save cleaned dataset
clean.to_csv("appointments_clean.csv", index=False)

# 9. Display results
print("appointments_clean.csv saved")
print("Appointments inserted:", len(clean))
print("Columns:", clean.columns.tolist())
print(clean.head())

appointments_clean.csv saved
Appointments inserted: 1020
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'lead_days', 'no_show']
  appointment_id patient_id clinic_id  booked_on  appt_date   age gender  \
0          A0001       P509       C05 2025-01-24 2025-01-25  60.0      F   
1          A0002       P046       C07 2025-05-27 2025-06-06  39.0      M   
2          A0003       P009       C02 2025-03-24 2025-03-30  63.0      F   
3          A0003       P009       C02 2025-03-24 2025-03-30  63.0      F   
4          A0004       P482       C01 2025-04-18 2025-04-22  52.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-in     Follow-up            No   
2          4.1      850         Walk-in  Consultation    

In [ ]:
# Remove duplicate appointment IDs
df = df.drop_duplicates(
    subset="appointment_id",
    keep="first"
).copy()

df = df.reset_index(drop=True)

print("Rows after removing duplicates:", len(df))
print("Duplicate IDs:", df["appointment_id"].duplicated().sum())

Rows after removing duplicates: 1000
Duplicate IDs: 0


In [ ]:
print("Rows:", len(df))
print("Duplicate IDs:", df["appointment_id"].duplicated().sum())
print("Missing values:")
print(df.isnull().sum())

Rows: 1000
Duplicate IDs: 0
Missing values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           6
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
lead_days           6
no_show             0
dtype: int64


In [ ]:
print("Missing age:", df["age"].isna().sum())
print("Missing distance:", df["distance_km"].isna().sum())
print("Missing appointment date:", df["appt_date"].isna().sum())
print("Missing status:", df["status"].isna().sum())

Missing age: 12
Missing distance: 20
Missing appointment date: 6
Missing status: 4


In [ ]:
import os

print(os.listdir())

['.config', '.ipynb_checkpoints', 'clinics.csv', 'appointments_clean.csv', 'appointments.csv', 'sample_data']


In [ ]:
import pandas as pd

clinics = pd.read_csv("clinics.csv")

print("Clinics rows:", len(clinics))
print(clinics.head())

Clinics rows: 8
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873


In [ ]:
clinics.to_csv("clinics_enriched.csv", index=False)
print("clinics_enriched.csv saved successfully")

clinics_enriched.csv saved successfully


In [ ]:

import pandas as pd

# 1. Start with the deduplicated dataframe
appointments = df.copy()

# 2. Select exactly 16 required columns
columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on",
    "appt_date", "age", "gender", "distance_km", "fee_inr",
    "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show"
]

appointments = appointments[columns].copy()

# 3. Remove rows with missing values
appointments = appointments.dropna(subset=columns)

# 4. Format dates
for col in ["booked_on", "appt_date"]:
    appointments[col] = pd.to_datetime(
        appointments[col], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

# Remove any rows with dates that could not be parsed
appointments = appointments.dropna(subset=["booked_on", "appt_date"])

# 5. Sort by appointment ID
appointments = appointments.sort_values(
    "appointment_id"
).reset_index(drop=True)

# 6. Validate
print("Rows:", len(appointments))
print("Unique IDs:", appointments["appointment_id"].nunique())
print("Duplicate IDs:", appointments["appointment_id"].duplicated().sum())
print("Missing values:", appointments.isna().sum().sum())
print("Columns:", len(appointments.columns))

# 7. Export CSV
appointments.to_csv("appointments_clean.csv", index=False)

# 8. Generate SQL statements
def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, (int, float)):
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"

sql_lines = []

# Clinics first
for _, row in clinics.iterrows():
    values = ", ".join(sql_value(v) for v in row.tolist())
    sql_lines.append(f"INSERT INTO clinics VALUES ({values});")

# Appointments second
for _, row in appointments.iterrows():
    values = ", ".join(sql_value(v) for v in row.tolist())
    sql_lines.append(f"INSERT INTO appointments VALUES ({values});")

with open("load_data.sql", "w", encoding="utf-8") as file:
    file.write("\n".join(sql_lines))

print("appointments_clean.csv saved")
print("load_data.sql saved")
print("Total SQL statements:", len(sql_lines))

Rows: 958
Unique IDs: 958
Duplicate IDs: 0
Missing values: 0
Columns: 16
appointments_clean.csv saved
load_data.sql saved
Total SQL statements: 966
